In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import math

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# Embedding
class Embed(nn.Module):
    def __init__(self, h=28, w=28, p=4, d=3):
        super().__init__()
        self.embed = nn.Conv2d(1, d, kernel_size=(p,p), stride=(p,p), padding=0)
        self.position_encoding = nn.Parameter(torch.zeros(h//p, w//p, d).float())
    def forward(self, x):
        x = self.embed(x) # B x 1 x h x w -> B x d x h//p x w//p
        x = x.permute(0, 2, 3, 1) # -> B x h//p x w//p x d
        x += self.position_encoding.unsqueeze(0)
        return x

In [ ]:
# Attention
class Attention(nn.Module):
    def __init__(self, d=3, num_heads=1):
        super(Attention, self).__init__()
        self.num_attention_heads = num_heads
        self.attention_head_size = d // self.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size # == d

        self.query = nn.Linear(d, self.all_head_size)
        self.key = nn.Linear(d, self.all_head_size)
        self.value = nn.Linear(d, self.all_head_size)

        self.out = nn.Linear(self.all_head_size, d)
        self.attn_dropout = nn.Dropout(0)
        self.proj_dropout = nn.Dropout(0)

        self.softmax = nn.Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_probs = self.softmax(attention_scores)
        weights = attention_probs
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights

In [ ]:
# SiLU
def swish(x):
    return x * torch.sigmoid(x)

In [ ]:
# perceptron
class Mlp(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 4*d)
        self.fc2 = nn.Linear(4*d, d)
        self.act_fn = swish
        self.dropout = nn.Dropout(0.1)
    def forward(self, x):
        x = self.fc1(x)
        x = self.act_fn(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [ ]:
# transformation
class Block(nn.Module):
    def __init__(self, n=50, d=3):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d)
        self.attn = Attention(d)
        self.ffn_norm = nn.LayerNorm(d)
        self.ffn = Mlp(d)
    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x += h
        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x += h
        return x, weights

In [ ]:
# wipeout
class Wipeout(nn.Module):
    def __init__(self, d=3, l=10):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.wipeout = nn.Linear(d, l, bias=False)
    def forward(self, x):
        return self.wipeout(self.norm(x)[:,0,:])

In [ ]:
# Transformer (encoder-only)
class Transformer(nn.Module):
    def __init__(self, h=28, w=28, p=4, d=3, num=6):
        super().__init__()
        self.embed = Embed(h,w,p,d)
        self.cls = nn.Parameter(torch.zeros(1,1,d).float())
        self.encoder = nn.ModuleList()
        for _ in range(num):
            self.encoder.append(Block((h//p)*(w//p)+1,d))
        self.wipeout = Wipeout(d)
    def forward(self, x):
        hidden_states = self.embed(x) # -> B x 7 x 7 x 3
        B, H, W, d = hidden_states.shape
        hidden_states = hidden_states.view(B,H*W,d) # -> B x 49 x 3
        cls_tokens = self.cls.expand(B, -1, -1)  # B x 1 x 3
        hidden_states = torch.cat([cls_tokens, hidden_states], dim=1) # -> B x 50 x 3
        attn_weights = []
        for block in self.encoder:
            hidden_states, weights = block(hidden_states)
            attn_weights.append(weights)
        return self.wipeout(hidden_states), torch.stack(attn_weights).permute(1,0,2,3,4) # B x 10, B x 6 x 1 x 50 x 50

In [ ]:
# create classifier
model = Transformer(28,28,4,16,6).to(device)

In [ ]:
# create train and test dataloader
batch_size = 512
train_loader = DataLoader(datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor()), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor()), batch_size=batch_size, shuffle=False)

In [ ]:
# Define loss funcion
criterion = nn.CrossEntropyLoss()

In [ ]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# scheduler
step_size = 16
gamma = 0.9
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
# training
history_loss = []
history_acc = []
history_test_acc = []
epoch = 0

In [ ]:
def train(num_epochs):
    global epoch
    for _ in range(num_epochs):
        # change model in training mood
        model.train()

        # to record loss and accuracy
        batch_loss = []
        total_train = 0
        correct_train = 0

        for batch, (x_train, y_train) in enumerate(train_loader):

            # send data to device
            input = x_train.to(device)

            # reset parameters gradient to zero
            optimizer.zero_grad()

            # forward pass to the model
            output, _ = model(input)

            # categorization
            expected_output = y_train.to(device)

            # cross entropy loss
            loss = criterion(output, expected_output)

            # find gradients
            loss.backward()
            # update parameters using gradients
            optimizer.step()

            # recording loss
            batch_loss.append(loss.item())

            # recording accuracy
            total_train += output.shape[0]
            correct_train += torch.argmax(output,dim=1).to('cpu').eq(y_train).sum().item()

        epoch_loss = np.average(batch_loss)
        epoch_acc = (100.0 * correct_train) / total_train

        history_loss.append(epoch_loss)
        history_acc.append(epoch_acc)

        total_test = 0
        correct_test = 0

        model.eval()

        for batch, (x_test, y_test) in enumerate(test_loader):

            # send data to device
            input = x_test.to(device)

            # forward pass to the model
            with torch.no_grad():
                output, _ = model(input)

            # recording accuracy
            total_test += output.shape[0]
            correct_test += torch.argmax(output,dim=1).to('cpu').eq(y_test).sum().item()

        test_acc = (100.0 * correct_test) / total_test

        history_test_acc.append(test_acc)

        print(f'Epoch: {epoch} Loss: {epoch_loss:.6f} Accuracy: {epoch_acc:.4f} Test accuracy: {test_acc:.4f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
        scheduler.step()
        epoch += 1

In [ ]:
# training
num_epochs = 40
print(f"number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
train(num_epochs)

In [ ]:
train(40)

In [ ]:
train(40)

In [ ]:
train(40)

In [ ]:
def plot_history():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # 1 row, 2 columns
    # Loss curve
    axes[0].plot(history_loss, 'r', linewidth=3.0)
    axes[0].set_title('Loss Curve', fontsize=16)
    axes[0].set_xlabel('Epochs', fontsize=14)
    axes[0].set_ylabel('Loss', fontsize=14)
    axes[0].legend(['Training Loss'], fontsize=12)
    # Accuracy curves
    axes[1].plot(history_acc, 'r', linewidth=3.0)
    axes[1].plot(history_test_acc, 'b', linewidth=3.0)
    axes[1].set_title('Accuracy Curves', fontsize=16)
    axes[1].set_xlabel('Epochs', fontsize=14)
    axes[1].set_ylabel('Accuracy', fontsize=14)
    axes[1].legend(['Training Accuracy', 'Validation Accuracy'], fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_history()

In [ ]:
# Save model and weights
def save():
    torch.save(model.state_dict(), classifier_name) # weights only
    print('Saved trained model at %s ' % classifier_name)

In [ ]:
classifier_name = 'mnist_ViT.pth'
save()

In [ ]:
# download model
from google.colab import files
files.download(classifier_name)

In [ ]:
# instead of trainig, we can download its result
#!wget http://agentspace.org/download/mnist_ViT.pth
#model.load_state_dict(torch.load("mnist_ViT.pth", map_location=torch.device(device)))

In [ ]:
# get few samples
test_iter = iter(test_loader)
x_sample, _ = next(test_iter)
print(x_sample.shape)

In [ ]:
# use the model on the samples
input_images = x_sample[0:100].to(device)
output_probabilities = model(input_images).to('cpu')
output_categories = [ category.item() for category in torch.argmax(output_probabilities, dim=1) ]
print(output_categories)

In [ ]:
# show results
def render(digit):
    img = np.zeros((28, 28), dtype=np.uint8)
    cv2.putText(img, str(digit), (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
    return img

plt.figure(figsize=(20, 40))
for b in range(10):
    for i in range(10):
        input_img = (x_sample[10*b+i].squeeze(0).detach().numpy()*255).astype(np.uint8)
        plt.subplot(20, 10, 20*b+i+1)
        plt.imshow(input_img, cmap='gray')
        plt.axis('off')
        output_img = render(output_categories[10*b+i])
        plt.subplot(20, 10, 20*b+i+1+10)
        plt.imshow(~output_img, cmap='gray')
        plt.axis('off')

plt.show()